# L5c Supporting Algorithm: Revised Simplex

This notebook is a selected solver-internals deeper dive. It supports the LP modeling work but is not a second required implementation lab.

> **Learning objectives**
>
> - Relate a basis to a corner of the feasible polytope.
> - Interpret reduced costs, entering variables, and the ratio test.
> - Explain why practical simplex performance can be excellent despite an exponential worst case.


## Setup

The local setup is loaded for consistency with every Week 5 notebook; this narrative does not call the solver.


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


## Historical algorithm narrative
The original Simplex Method, invented by George Dantzig in 1947, is an iterative algorithm that solves a linear program by moving around the collection of corner points of the feasible _polytope_, improving the objective at each step until no further gain is possible.

> __Myth or fact?__ As a graduate student in 1939, Dantzig once arrived late to a statistics lecture, mistook two well-known unsolved problems on the blackboard for homework, and solved them over the next few days, before realizing they were open research questions! That story is true, but it happened years before he developed the simplex method and did not directly inspire the algorithm. But still, it's a fun story; being late isn't always bad!

__Simplex is a big deal__: Before simplex, LPs were mostly a theoretical curiosity; afterwards, they became essential tools, radically improving resource allocation and strategic planning in the second half of the twentieth century. Over seventy years later, the simplex method remains widely used in commercial optimization software. This is despite some not-so-great worst-case performance bounds!

### Bases and an initial feasible solution

We are going to focus on the _revised simplex algorithm_. The method partitions the variables into a __basic set__ and a __nonbasic set__. We hold the nonbasic variables at zero and determine the basic variables from the constraints. Changing which variables are basic allows us to search for a solution with a lower objective value. Let's first examine how this partition defines a feasible starting point.

Suppose we have $n$ decision variables collected in $x$, $m$ inequality constraints with coefficient matrix $A\in\mathbb{R}^{m\times n}$ and right-hand side $b\in\mathbb{R}^{m}$, and objective coefficients $c\in\mathbb{R}^{n}$. We assume $b\geq 0$. Introducing a vector of slack variables $s\in\mathbb{R}^{m}$ gives the minimization problem:

$$
\begin{aligned}
\text{minimize}\quad & c^\top x \\
\text{subject to}\quad & Ax+s=b, \\
& x\geq 0,\quad s\geq 0.
\end{aligned}
$$

Each slack variable measures the difference between a constraint's upper bound and its left-hand side. Thus, $Ax+s=b$ with $s\geq 0$ is equivalent to $Ax\leq b$. If the original problem maximizes an objective, we negate its coefficients to obtain this minimization form.

Both decision variables and slack variables can enter the basis, so our matrix must contain columns for both. Collect the variables, constraint columns, and objective coefficients as follows:

$$
z=\begin{bmatrix}x\\s\end{bmatrix},\qquad
\widetilde A=\begin{bmatrix}A&I_m\end{bmatrix},\qquad
\widetilde c=\begin{bmatrix}c\\0_m\end{bmatrix},
$$

where $I_m$ is the $m\times m$ identity matrix and $0_m$ is a vector of $m$ zeros. The augmented matrix $\widetilde A$ has $n+m$ columns. The slack variables have zero objective coefficients, so the constraints become $\widetilde A z=b$ with $z\geq 0$, and the objective remains $\widetilde c^\top z=c^\top x$.

> __How does a basis identify a corner?__
>
> Choose $m$ linearly independent columns of $\widetilde A$. Let $B=(B_1,\ldots,B_m)$ be their ordered list of column indices, and let $N$ contain the remaining indices. The selected columns form the invertible __basis matrix__ $\widetilde A_B$. Setting the nonbasic variables to zero determines the basic variables through the linear system:
>
> $$
> z_N=0,\qquad \widetilde A_B z_B=b.
> $$
>
> If the resulting basic values satisfy $z_B\geq 0$, we have a __basic feasible solution__, which represents a corner of the feasible region. A choice of independent columns alone does not guarantee feasibility; the solution must also satisfy nonnegativity.
>
> A basic variable can be zero. When at least one basic variable is zero, the basic feasible solution is __degenerate__, and different bases may represent the same corner. Thus, exchanging basic and nonbasic variables need not move us to a different point or improve the objective.

This distinction separates the variables we solve for from the values they take. The basis selects the columns used in the linear system; feasibility depends on the resulting values.

__Initialization.__ The slack columns give us a convenient first basis. With the entries of $z$ ordered as above, choose the initial indices as follows:

$$
B=(n+1,\ldots,n+m),\qquad N=\{1,\ldots,n\}.
$$

The initial basis matrix is $\widetilde A_B=I_m$. Setting the decision variables to zero and solving for the slacks gives:

$$
x^{(0)}=0,\qquad s^{(0)}=b.
$$

Because $b\geq 0$, this starting solution is feasible. If any component of $b$ is zero, the initial basic feasible solution is degenerate. This initialization relies on the stated inequality form and nonnegative right-hand side; other formulations may require a separate procedure to find a feasible basis.

Set the pivot counter $t=0$ and choose a positive integer limit $T$ on the number of pivots. We can now examine how reduced costs identify a candidate entering variable and how the ratio test determines the feasible step.


### Reduced costs and feasible steps

Starting from a basic feasible solution, we want to determine whether increasing a nonbasic variable can lower the objective. Its objective coefficient alone does not answer this question: increasing that variable also requires changes in the basic variables to maintain the equality constraints. The __reduced cost__ accounts for both contributions.

Let $\lambda\in\mathbb{R}^{m}$ be the multipliers associated with the current basis, and let $\mu_i$ be the reduced cost of nonbasic variable $z_i$. We calculate these quantities by solving for $\lambda$ and then evaluating each reduced cost:

$$
\begin{aligned}
\widetilde A_B^\top\lambda&=\widetilde c_B,\\
\mu_i&=\widetilde c_i-\lambda^\top\widetilde A_i,\qquad i\in N,
\end{aligned}
$$

where $\widetilde A_i$ is column $i$ of the augmented constraint matrix and $\widetilde c_B$ contains the objective coefficients of the current basic variables. We write the multiplier calculation as a linear system; an explicit matrix inverse is not required.

__Optimality test.__ Using $\widetilde A z=b$ and the multiplier equation, we can express the objective of any feasible solution in terms of its nonbasic variables:

$$
\widetilde c^\top z
=\lambda^\top b+\sum_{i\in N}\mu_i z_i.
$$

At the current basic feasible solution, $z_N=0$, so its objective is $\lambda^\top b$. If every nonbasic reduced cost is nonnegative, the sum cannot be negative at any feasible solution because $z_i\geq 0$. The current solution therefore attains the minimum, and we stop with an __optimal__ status.

Otherwise, choose an entering index $e$ with the most negative reduced cost. This is __Dantzig's entering-variable rule__, written as:

$$
e\in\underset{i\in N}{\operatorname{arg\,min}}\;\mu_i,\qquad \mu_e<0.
$$

We choose one minimizing index if there is a tie. A negative reduced cost identifies a candidate direction; we still need to determine whether a positive step is feasible.

__Direction and objective change.__ Let $\alpha\geq 0$ be the proposed increase in the entering variable, which currently has value zero. To determine the corresponding change $-\alpha d$ in the basic variables, solve the following linear system for $d\in\mathbb{R}^{m}$:

$$
\widetilde A_B d=\widetilde A_e.
$$

If we increase $z_e$ by $\alpha$, the current basic variables must change as follows:

$$
z_e(\alpha)=\alpha,\qquad z_B(\alpha)=z_B-\alpha d.
$$

All other nonbasic variables remain zero. These changes preserve the equality constraints because $\widetilde A_B(z_B-\alpha d)+\widetilde A_e\alpha=b$. The change in the objective is therefore:

$$
\begin{aligned}
\Delta f
&=\alpha\widetilde c_e-\alpha\widetilde c_B^\top d\\
&=\alpha\bigl(\widetilde c_e-\lambda^\top\widetilde A_e\bigr)\\
&=\alpha\mu_e.
\end{aligned}
$$

Thus, the objective strictly decreases when $\mu_e<0$ and the feasible step satisfies $\alpha>0$. A zero step leaves the objective unchanged.

__Ratio test.__ We must also keep the basic variables nonnegative. For a position $j$ in the ordered basis, a positive component $d_j$ makes $(z_B)_j$ decrease as $\alpha$ increases. That variable reaches zero at $\alpha=(z_B)_j/d_j$. When at least one $d_j>0$, the largest feasible step is the first of these limits:

$$
\alpha^\star=\min_{j:\,d_j>0}\frac{(z_B)_j}{d_j},\qquad
\ell\in\underset{j:\,d_j>0}{\operatorname{arg\,min}}\;\frac{(z_B)_j}{d_j}.
$$

The index $\ell$ is a position in the current basis; the corresponding variable $z_{B_\ell}$ leaves the basis. Components with $d_j\leq 0$ impose no upper limit on the step because those basic values remain constant or increase.

If no component of $d$ is positive, every $\alpha\geq 0$ is feasible along this direction. Since $\mu_e<0$, the objective decreases without bound, and we stop with an __unbounded__ status.

If the ratio test gives $\alpha^\star=0$, the pivot is __degenerate__: the basis changes, but the solution and objective do not. This is why a negative reduced cost alone does not establish that a degenerate current solution is nonoptimal. With the entering variable, leaving position, and step determined, we can now update the solution and basis in the correct order.


### Pivot and stopping conditions

Suppose the ratio test gives a finite step $\alpha^\star$ and a leaving position $\ell$. We must update the solution using the __current__ basis before replacing its leaving index. Otherwise, the entries of the direction vector $d$ would be paired with the wrong variables.

First, save the current basic and nonbasic indices and the index $r$ of the leaving variable:

$$
B^{\mathrm{old}}\gets B,\qquad
N^{\mathrm{old}}\gets N,\qquad
r\gets B^{\mathrm{old}}_\ell.
$$

The vector $z^{(t)}$ is the current solution after $t$ completed pivots. Form the next solution using the saved indices:

$$
\begin{aligned}
z_{B^{\mathrm{old}}}^{(t+1)}&=z_{B^{\mathrm{old}}}^{(t)}-\alpha^\star d,\\
z_e^{(t+1)}&=\alpha^\star,\\
z_i^{(t+1)}&=0,\qquad i\in N^{\mathrm{old}}\setminus\{e\}.
\end{aligned}
$$

The ratio test makes the leaving variable satisfy $z_r^{(t+1)}=0$, while preserving nonnegativity of every variable. We can therefore hold $z_r$ at zero as a nonbasic variable and replace its basis column with the entering column. Update the ordered basis, nonbasic set, and counter as follows:

$$
\begin{aligned}
B_j&\gets
\begin{cases}
e,&j=\ell,\\
B_j^{\mathrm{old}},&j\ne\ell,
\end{cases}
\qquad j=1,\ldots,m,\\[4pt]
N&\gets\bigl(N^{\mathrm{old}}\setminus\{e\}\bigr)\cup\{r\},\\
t&\gets t+1.
\end{aligned}
$$

The positive pivot component $d_\ell$ ensures that the new basis matrix is invertible. The next iteration uses this new basis to recompute the multipliers and reduced costs. A degenerate pivot follows the same update: when $\alpha^\star=0$, the basis changes even though the solution does not.

__Stopping conditions.__ At the start of each iteration, test the newly computed reduced costs for optimality. If the test fails and $t=T$, stop because the allowed number of pivots has been reached. Otherwise, choose the entering variable and compute its direction. An unbounded direction ends the calculation; a finite ratio-test step leads to the next pivot.

> __What does the returned status mean?__
>
> * **Optimal:** The current solution is feasible and every nonbasic reduced cost is nonnegative. Together, these conditions certify that the objective has reached its minimum.
> * **Unbounded:** A variable with negative reduced cost has a direction with $d_j\leq 0$ for every basic position $j$. We can increase that variable indefinitely while remaining feasible, driving the objective toward $-\infty$.
> * **Iteration limit:** We have completed $T$ pivots without obtaining an optimality certificate. The current solution is feasible, but whether it is optimal remains unresolved.

Testing optimality before the pivot limit allows us to recognize a solution that becomes optimal on the final permitted pivot. Reaching the limit alone does not establish convergence.

__Degeneracy and cycling.__ A sequence of zero-step pivots can revisit a previous basis, a behavior called _cycling_. Dantzig's entering-variable rule does not prevent this, so the iteration limit may be reached without finding an optimality certificate. One way to prevent cycling is [Bland's rule](https://ocw.mit.edu/courses/6-251j-introduction-to-mathematical-programming-fall-2009/2ef2f1dd7045b5f29e5faea299fb0798_MIT6_251JF09_lec06.pdf): choose the smallest-index nonbasic variable with negative reduced cost and, among tied minimum ratios, choose the basic variable with the smallest variable index to leave. This changes both selection rules and guarantees finite termination in exact arithmetic. Our sketch retains Dantzig's entering rule; Bland's rule explains how an anti-cycling safeguard can change the selection procedure.

We now have the complete sequence: test optimality, determine a feasible step or identify unboundedness, and update the solution and basis while tracking the number of pivots.


Wow! That seems intense. How efficient is the simplex algorithm?
* In the __worst case__, the simplex method can take _exponential time_ in the number of variables. Klee and Minty's 1972 example shows it may visit all $2^n$ vertices of an $n$-dimensional cube, forcing on the order of $2^n$ pivots. Thus, it has $O(2^n)$ worst-case complexity.
* However, __in practice__, the simplex method is often very efficient. It performs well on most real-world problems, and its average-case performance is polynomial time for many practical instances. The worst-case exponential bound is rarely encountered in practice, as most LPs have a structure that allows the simplex method to converge quickly.

For Week 5, stop here: interior-point derivations are reserved for deeper study.

___


## Summary

Revised simplex stores and updates a basis instead of enumerating every corner. Reduced costs answer whether a nonbasic variable can improve the objective; the ratio test chooses a feasible step and leaving variable. Those mechanics help explain solver output, but students are not expected to implement a production LP solver this week.
